# E-005 — Línea base tabular de HVI (EchoNext)

**Para qué sirve:** fija el *piso* que el modelo de ECG tiene que superar. Si una regresión logística con edad, sexo e intervalos del ECG (FC, PR, QRS, QTc) ya logra cierto AUC, la red neuronal solo aporta algo si lo supera. Registro en Obsidian: *Experimentos — Modelo IA* → E-005; *Decisiones* → D-006.

**Datos:** `MyDrive/modelo-hvi/datos/echonext/`. Licencia restringida de PhysioNet: no compartir la carpeta ni copiarla al vault.

**Salida:** solo agregados → `MyDrive/modelo-hvi/resultados/E-005_linea_base_tabular.json`.

**Reglas:**
- Nunca imprimir filas de pacientes.
- Antes de compartir: *Editar → Borrar todos los resultados*.
- El código se versiona en `github.com/luchoaquinopla/modelo-hvi` (`notebooks/`).

**Cómo correrlo:** *Entorno de ejecución → Ejecutar todas*. Alcanza con CPU (~1 min).

In [ ]:
# Carga de la metadata de EchoNext (datos restringidos: solo Drive privado, nunca el vault)
import subprocess
import sys
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
CARPETA = Path('/content/drive/MyDrive/modelo-hvi')
RUTA_METADATA = CARPETA / 'datos' / 'echonext' / 'echonext_metadata_100k.csv'

if not RUTA_METADATA.exists():
    # Alternativa: S3 de PhysioNet con clave en Colab Secrets (borrarla al terminar)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'boto3'], check=True)
    import boto3
    from google.colab import userdata
    s3 = boto3.client(
        's3', region_name='us-east-1',
        aws_access_key_id=userdata.get('AWS_ACCESS_KEY_ID'),
        aws_secret_access_key=userdata.get('AWS_SECRET_ACCESS_KEY'),
    )
    PUNTO_DE_ACCESO = 'arn:aws:s3:us-east-1:724665945834:accesspoint/echonext-v1-1-1-01'
    claves = [o['Key'] for o in s3.list_objects_v2(Bucket=PUNTO_DE_ACCESO, Prefix='echonext/1.1.1/')['Contents']]
    clave = next(k for k in claves if k.lower().endswith('metadata_100k.csv'))
    RUTA_METADATA.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(PUNTO_DE_ACCESO, clave, str(RUTA_METADATA))

metadata = pd.read_csv(RUTA_METADATA)
print(metadata.shape)
print(metadata.split.value_counts().to_dict())

In [ ]:
# E-005: línea base tabular (D-006). Solo se exportan agregados.
import json
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEMILLA = 20260912
ETIQUETA = 'lvwt_gte_13_flag'
CONTINUAS = ['age_at_ecg', 'ventricular_rate', 'pr_interval', 'qrs_duration', 'qt_corrected']
# atrial_rate queda afuera: no está en el PDF del instituto (D-007)
MODELOS = {
    'demografico': ['age_at_ecg', 'sexo_masculino'],
    'tabular': ['age_at_ecg', 'sexo_masculino', 'ventricular_rate', 'pr_interval', 'qrs_duration', 'qt_corrected'],
}
VALORES_C = [0.001, 0.01, 0.1, 1.0, 10.0]
TRAMOS_EDAD = [18, 40, 50, 60, 70, 80, 91]

datos = metadata[metadata.split.isin(['train', 'val', 'test'])].copy()
datos['sexo_masculino'] = datos.sex.map({'male': 1, 'female': 0, 1: 1, 0: 0})
assert datos.sexo_masculino.notna().all(), 'sexo con valores inesperados'
datos['tramo_edad'] = pd.cut(datos.age_at_ecg, TRAMOS_EDAD, right=False).astype(str)
particiones = {p: datos[datos.split == p] for p in ['train', 'val', 'test']}
entrenamiento = particiones['train']
# Cada paciente pesa lo mismo en train (hasta 146 ECG por paciente, E-003)
peso_por_paciente = 1 / entrenamiento.groupby('patient_key').patient_key.transform('size')


def armar_modelo(columnas, c):
    continuas = [col for col in columnas if col in CONTINUAS]
    preprocesamiento = ColumnTransformer(
        [('continuas', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), continuas)],
        remainder='passthrough',
    )
    return make_pipeline(preprocesamiento, LogisticRegression(C=c, max_iter=2000))


def entrenar(columnas, c):
    modelo = armar_modelo(columnas, c)
    modelo.fit(entrenamiento[columnas], entrenamiento[ETIQUETA], logisticregression__sample_weight=peso_por_paciente)
    return modelo


def metricas(y, p, umbral):
    y, p = np.asarray(y), np.asarray(p)
    predicho = p >= umbral
    vp, fn = int((predicho & (y == 1)).sum()), int((~predicho & (y == 1)).sum())
    vn, fp = int((~predicho & (y == 0)).sum()), int((predicho & (y == 0)).sum())
    return {
        'n': int(len(y)), 'positivos': int(y.sum()),
        'auc': round(float(roc_auc_score(y, p)), 4) if 0 < y.sum() < len(y) else None,
        'sensibilidad': round(vp / (vp + fn), 4) if vp + fn else None,
        'especificidad': round(vn / (vn + fp), 4) if vn + fp else None,
    }


def bootstrap_auc(y, predicciones, repeticiones=1000):
    # Remuestreo pareado: mismo índice para todos los modelos
    rng = np.random.default_rng(SEMILLA)
    y = np.asarray(y)
    valores = {nombre: [] for nombre in predicciones}
    for _ in range(repeticiones):
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(y):
            for nombre, p in predicciones.items():
                valores[nombre].append(roc_auc_score(y[i], np.asarray(p)[i]))
    return {nombre: np.array(v) for nombre, v in valores.items()}


def intervalo(v):
    return [round(float(np.percentile(v, 2.5)), 4), round(float(np.percentile(v, 97.5)), 4)]


resultados = {
    'semilla': SEMILLA, 'etiqueta': ETIQUETA,
    'faltantes_pct': {p: (df[CONTINUAS].isna().mean() * 100).round(2).to_dict() for p, df in particiones.items()},
    'modelos': {},
}
predicciones_test = {}
validacion, prueba = particiones['val'], particiones['test']

for nombre, columnas in MODELOS.items():
    auc_val = {c: roc_auc_score(validacion[ETIQUETA], entrenar(columnas, c).predict_proba(validacion[columnas])[:, 1]) for c in VALORES_C}
    mejor_c = max(auc_val, key=auc_val.get)
    modelo = entrenar(columnas, mejor_c)
    p_val = modelo.predict_proba(validacion[columnas])[:, 1]
    fpr, tpr, umbrales = roc_curve(validacion[ETIQUETA], p_val)
    umbral = float(umbrales[np.argmax(tpr - fpr)])  # Youden, fijado en val
    p_test = modelo.predict_proba(prueba[columnas])[:, 1]
    predicciones_test[nombre] = p_test
    coeficientes = dict(zip(modelo[:-1].get_feature_names_out(), modelo[-1].coef_[0].round(4).tolist()))
    resultados['modelos'][nombre] = {
        'columnas': columnas, 'c_elegido': mejor_c,
        'auc_val_por_c': {str(c): round(float(v), 4) for c, v in auc_val.items()},
        'umbral_youden_val': round(umbral, 4),
        'coeficientes_estandarizados': coeficientes,
        'test': {
            **metricas(prueba[ETIQUETA], p_test, umbral),
            'auprc': round(float(average_precision_score(prueba[ETIQUETA], p_test)), 4),
            'brier': round(float(brier_score_loss(prueba[ETIQUETA], p_test)), 4),
        },
        'test_por_sexo': {s: metricas(g[ETIQUETA], p_test[prueba.sex == s], umbral) for s, g in prueba.groupby('sex')},
        'test_por_edad': {t: metricas(g[ETIQUETA], p_test[prueba.tramo_edad == t], umbral) for t, g in prueba.groupby('tramo_edad')},
    }

muestras = bootstrap_auc(prueba[ETIQUETA], predicciones_test)
for nombre in MODELOS:
    resultados['modelos'][nombre]['test']['auc_ic95'] = intervalo(muestras[nombre])
resultados['diferencia_auc_tabular_menos_demografico'] = {
    'media': round(float((muestras['tabular'] - muestras['demografico']).mean()), 4),
    'ic95': intervalo(muestras['tabular'] - muestras['demografico']),
}

ruta_resultados = CARPETA / 'resultados' / 'E-005_linea_base_tabular.json'
ruta_resultados.parent.mkdir(parents=True, exist_ok=True)
ruta_resultados.write_text(json.dumps(resultados, indent=2, ensure_ascii=False))
print(json.dumps(resultados, indent=2, ensure_ascii=False))